<a href="https://colab.research.google.com/github/nyp-sit/iti121-2026s2/blob/main/L3/04a_asl_augmentation_new.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Applied Deep Learning
## 4a. Data Augmentation



In [ ]:
# Public dataset access
# If kagglehub is not installed in a new environment, install it once with:
# %pip install -q kagglehub
from pathlib import Path
import os
import kagglehub

dataset_path = Path(
    kagglehub.dataset_download("datamunge/sign-language-mnist")
)

print("Dataset folder:", dataset_path)


# 4a. Data Augmentation

So far, we've selected a model architecture that improves image-classification performance because it is designed to recognise important spatial features in images. However, validation performance may still lag behind training performance, which is a common sign of **overfitting**: the model performs well on examples it has already seen but generalises less well to unseen data.

To make the model more robust, we can programmatically increase the **variation** in the training data. This is known as **data augmentation**, a useful technique in many deep-learning applications.

Augmentation does not simply duplicate images. Instead, it creates modified versions of training images so that the model is encouraged to focus on features that remain useful across realistic changes in orientation, crop, brightness, contrast, and other conditions.

## 4a.1 Objectives

* Augment the ASL dataset
* Use the augmented data to train an improved model
* Save the trained model parameters to disk for use in the next practical

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as transforms
import torchvision.transforms.functional as F
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch version:", torch.__version__)
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 4a.2 Preparing the Data

As we're in a new notebook, we will load and process our data again.

This independent version uses the public **Sign Language MNIST** dataset from Kaggle. The static-image dataset contains 24 ASL letter classes; **J** and **Z** are excluded because those signs involve motion.

In [ ]:
IMG_HEIGHT = 28
IMG_WIDTH = 28
IMG_CHS = 1

TRAIN_CSV = dataset_path / "sign_mnist_train" / "sign_mnist_train.csv"
VALID_CSV = dataset_path / "sign_mnist_test" / "sign_mnist_test.csv"

# Some kagglehub versions expose the CSV files directly in the dataset folder.
if not TRAIN_CSV.exists():
    TRAIN_CSV = dataset_path / "sign_mnist_train.csv"
if not VALID_CSV.exists():
    VALID_CSV = dataset_path / "sign_mnist_test.csv"

train_df = pd.read_csv(TRAIN_CSV)
valid_df = pd.read_csv(VALID_CSV)

# Public Sign Language MNIST uses alphabet-position labels with J and Z absent.
# Remap the remaining raw labels to contiguous class indices 0 ... 23.
unique_labels = sorted(train_df["label"].unique())
label_to_index = {label: idx for idx, label in enumerate(unique_labels)}
index_to_label = {idx: label for label, idx in label_to_index.items()}

train_df = train_df.copy()
valid_df = valid_df.copy()
train_df["label"] = train_df["label"].map(label_to_index)
valid_df["label"] = valid_df["label"].map(label_to_index)

N_CLASSES = len(unique_labels)


class MyDataset(Dataset):
    def __init__(self, base_df):
        x_df = base_df.copy()
        y_df = x_df.pop("label")

        x_df = x_df.values.astype("float32") / 255.0
        x_df = x_df.reshape(-1, IMG_CHS, IMG_HEIGHT, IMG_WIDTH)

        # Keep the complete dataset in CPU memory.
        # Mini-batches are moved to the GPU during training.
        self.xs = torch.tensor(x_df, dtype=torch.float32)
        self.ys = torch.tensor(y_df.values, dtype=torch.long)

    def __getitem__(self, idx):
        return self.xs[idx], self.ys[idx]

    def __len__(self):
        return len(self.xs)


n = 32

train_data = MyDataset(train_df)
train_loader = DataLoader(
    train_data,
    batch_size=n,
    shuffle=True,
    num_workers=0
)
train_N = len(train_loader.dataset)

valid_data = MyDataset(valid_df)
valid_loader = DataLoader(
    valid_data,
    batch_size=n,
    shuffle=False,
    num_workers=0
)
valid_N = len(valid_loader.dataset)

print("Number of classes :", N_CLASSES)
print("Training samples  :", train_N)
print("Validation samples:", valid_N)


## 4a.3 Model Creation

We will also need to create our model again. As we learned in the previous practical, convolutional neural networks often use repeated sequences of layers. We can take advantage of this pattern by defining our own **custom module** and then using that module inside a `Sequential` model.

To do this, we extend PyTorch's `nn.Module` class and define two methods:

* `__init__`: defines the layers and properties used by the module.
* `forward`: defines how input data passes through the module.

This allows us to treat a repeated CNN block almost like a single layer.

In [ ]:
class MyConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout_p):
        kernel_size = 3
        super().__init__()

        self.model = nn.Sequential(
            nn.Conv2d(
                in_ch,
                out_ch,
                kernel_size,
                stride=1,
                padding=1
            ),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.MaxPool2d(2, stride=2)
        )

    def forward(self, x):
        return self.model(x)


Now that we've defined our custom module, let's use it in the same CNN architecture as the previous practical. Can you see how each `MyConvBlock` replaces a repeated sequence of convolution, normalisation, activation, dropout, and pooling operations?

In [ ]:
flattened_img_size = 75 * 3 * 3

# Input: 1 x 28 x 28
base_model = nn.Sequential(
    MyConvBlock(IMG_CHS, 25, 0),      # 25 x 14 x 14
    MyConvBlock(25, 50, 0.2),         # 50 x 7 x 7
    MyConvBlock(50, 75, 0),           # 75 x 3 x 3

    # Flatten to dense layers
    nn.Flatten(),
    nn.Linear(flattened_img_size, 512),
    nn.Dropout(0.3),
    nn.ReLU(),
    nn.Linear(512, N_CLASSES)
)


When we print the model, it shows both the use of our custom module and the layers inside each block.

In [ ]:
loss_function = nn.CrossEntropyLoss()
optimizer = Adam(base_model.parameters())

# Keep normal eager-mode execution for portability across
# local Windows/Jupyter and Google Colab environments.
model = base_model.to(device)

model


Custom modules are flexible, and we can define additional methods or properties when needed. This makes them useful for organising repeated or more complex neural-network structures.

## 4a.4 Data Augmentation

Before defining our training loop, it's time to set up data augmentation.

We will use **TorchVision transforms** to create realistic variations of the training images. First, let's take one sample image so that we can inspect each transformation separately.

In [ ]:
row_0 = train_df.head(1).copy()
y_0 = row_0.pop("label")

x_0 = row_0.values.astype("float32") / 255.0
x_0 = x_0.reshape(IMG_CHS, IMG_HEIGHT, IMG_WIDTH)
x_0 = torch.tensor(x_0, dtype=torch.float32)

x_0.shape


In [ ]:
image = F.to_pil_image(x_0)
plt.imshow(image, cmap="gray")
plt.axis("off")
plt.show()


### 4a.4.1 `RandomResizedCrop`

This transform randomly resizes the input image according to a specified `scale`, then crops it back to the requested output size.

Here, we crop back to the original `28 × 28` dimensions. Since the image is square, we keep the aspect-ratio range fixed at `1:1`.

In [ ]:
trans = transforms.Compose([
    transforms.RandomResizedCrop(
        (IMG_HEIGHT, IMG_WIDTH),
        scale=(0.7, 1.0),
        ratio=(1.0, 1.0)
    ),
])


Try running the cell below a few times. The crop should vary because the transformation is random.

In [ ]:
new_x_0 = trans(x_0)

image = F.to_pil_image(new_x_0)
plt.imshow(image, cmap="gray")
plt.axis("off")
plt.show()


In [ ]:
new_x_0.shape

### 4a.4.2 `RandomHorizontalFlip`

We can also randomly flip images horizontally or vertically. For these ASL images, we will only consider a **horizontal** flip.

Take a moment to think about why horizontal flipping may be reasonable, while vertical flipping would usually create unrealistic examples.

**Solution:** ASL signs can be performed with either the left or right hand being dominant. A horizontal flip can therefore represent a plausible change in handedness. An upside-down hand sign, however, would generally not represent a realistic example.

This is an example of why **domain knowledge matters when selecting augmentation techniques**.

In [ ]:
trans = transforms.Compose([
    transforms.RandomHorizontalFlip()
])


Try running the cell below a few times. Does the image flip approximately half the time?

In [ ]:
new_x_0 = trans(x_0)

image = F.to_pil_image(new_x_0)
plt.imshow(image, cmap="gray")
plt.axis("off")

plt.show()


### 4a.4.3 `RandomRotation`

We can also randomly rotate the image to add more variation.

As with other augmentation techniques, it is possible to go too far. Excessive rotation can change the visual meaning of a hand sign. We therefore use only a small rotation range.

In [ ]:
trans = transforms.Compose([
    transforms.RandomRotation(10)
])


When an image is rotated, empty pixels can appear around the corners because some original pixels move outside the image boundary.

In [ ]:
new_x_0 = trans(x_0)

image = F.to_pil_image(new_x_0)
plt.imshow(image, cmap="gray")
plt.axis("off")
plt.show()


### 4a.4.4 `ColorJitter`

`ColorJitter` can randomly vary:

* brightness;
* contrast;
* saturation;
* hue.

Because our ASL images are grayscale, we will focus on **brightness** and **contrast**.

In [ ]:
brightness = 0.2  # Try values from 0 to 1
contrast = 0.5      # Try values from 0 to 1

trans = transforms.Compose([
    transforms.ColorJitter(
        brightness=brightness,
        contrast=contrast
    )
])


Run the cell below a few times. Also try changing either `brightness` or `contrast` to `1`. What happens to the image?

In [ ]:
new_x_0 = trans(x_0)

image = F.to_pil_image(new_x_0)
plt.imshow(image, cmap="gray")
plt.axis("off")
plt.show()


### 4a.4.5 `Compose`

Time to bring the transformations together. `Compose` allows us to apply a sequence of random transformations as one augmentation pipeline.

In [ ]:
random_transforms = transforms.Compose([
    transforms.RandomRotation(5),
    transforms.RandomResizedCrop(
        (IMG_HEIGHT, IMG_WIDTH),
        scale=(0.9, 1.0),
        ratio=(1.0, 1.0)
    ),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.5
    )
])


Let's test the complete pipeline. Because several transformations contain random choices, a single source image can generate a very large number of slightly different training examples.

In [ ]:
new_x_0 = random_transforms(x_0)

image = F.to_pil_image(new_x_0)
plt.imshow(image, cmap="gray")
plt.axis("off")
plt.show()


### 4a.5 Training with Augmentation

The training loop is almost the same as before, with one important change:

```python
x = random_transforms(x)
```

The random transformations are applied **only to the training images** before they are passed to the model.

We calculate accuracy directly in this notebook so that there is no dependency on an external `utils.py` file.

In [ ]:
def train():
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    model.train()

    for x, y in train_loader:
        # Augment while the batch is still on CPU.
        x = random_transforms(x)

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        output = model(x)
        batch_loss = loss_function(output, y)

        batch_loss.backward()
        optimizer.step()

        batch_size = y.size(0)
        total_loss += batch_loss.item() * batch_size
        total_correct += (output.argmax(dim=1) == y).sum().item()
        total_samples += batch_size

    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples

    print(
        "Train - Loss: {:.4f} Accuracy: {:.4f}".format(
            avg_loss,
            accuracy
        )
    )

    return avg_loss, accuracy


Validation remains unchanged: **no random augmentation is applied to the validation data**.

The validation set should represent the original unseen data distribution as consistently as possible.

In [ ]:
def validate():
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    model.eval()

    with torch.no_grad():
        for x, y in valid_loader:
            x = x.to(device)
            y = y.to(device)

            output = model(x)
            batch_loss = loss_function(output, y)

            batch_size = y.size(0)
            total_loss += batch_loss.item() * batch_size
            total_correct += (output.argmax(dim=1) == y).sum().item()
            total_samples += batch_size

    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples

    print(
        "Valid - Loss: {:.4f} Accuracy: {:.4f}".format(
            avg_loss,
            accuracy
        )
    )

    return avg_loss, accuracy


Let's put data augmentation to the test.

In [ ]:
epochs = 1
history = []

for epoch in range(1, epochs + 1):
    print("Epoch: {}".format(epoch))

    train_loss, train_acc = train()
    valid_loss, valid_acc = validate()

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "valid_loss": valid_loss,
        "valid_acc": valid_acc
    })


## Discussion of Results

Compare the training and validation results with the previous CNN practical.

With useful augmentation, validation performance should often become **more stable** and the gap between training and validation performance may decrease. Training accuracy can sometimes be lower because the model is being asked to learn from continually changing versions of the training images.

However, augmentation does not guarantee improvement. The result depends on whether the selected transformations produce examples that are **realistic for the problem**.

### Think

- Did the training accuracy decrease?
- Did validation accuracy become more stable?
- Did the gap between training and validation accuracy change?
- Which augmentation would become harmful if made too aggressive?


## Saving the Model

Now that we have a trained model, we will save it so that it can be loaded in the next practical for inference on new images.

Rather than serialising the complete Python model object, this independent version saves the model's **`state_dict`** together with a small amount of metadata. This is more portable and avoids depending on a pickled copy of the original Python class object.

To reload the model later, we recreate the architecture and then load the saved parameter values.

In [ ]:
CHECKPOINT_PATH = Path.cwd() / "asl_cnn_augmented_checkpoint.pth"

checkpoint = {
    "model_state_dict": base_model.state_dict(),
    "n_classes": N_CLASSES,
    "img_height": IMG_HEIGHT,
    "img_width": IMG_WIDTH,
    "img_channels": IMG_CHS,
    "unique_raw_labels": unique_labels
}

torch.save(checkpoint, CHECKPOINT_PATH)

print("Saved checkpoint to:")
print(CHECKPOINT_PATH)


## Summary

In this section, you used TorchVision to augment the ASL training dataset.

You:

- inspected several augmentation techniques individually;
- combined them using `Compose`;
- applied augmentation only during training;
- compared training and validation behaviour;
- saved the trained CNN parameters for later inference.

The key idea is that **good augmentation introduces realistic variation without changing the meaning of the class**.

## Next

Now that you have a trained model saved to disk, the next practical will load the checkpoint and use the CNN to make predictions on previously unseen ASL images.

Continue to the **Model Predictions** practical.

---
### Dataset attribution

**Sign Language MNIST** — Kaggle dataset by *datamunge*  
License: **CC0: Public Domain**

This notebook is an independent Applied Deep Learning teaching resource and does not depend on NVIDIA DLI course infrastructure, hosted course files, or helper modules.
